In [15]:
import sys
print(sys.executable)

c:\Users\ASUS\anaconda3\envs\deepfer\python.exe


In [16]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [17]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

df = pd.read_csv('tourism_cleaned_engineered.csv')

# IMPORTANT: drop the leak-prone columns computed in the EDA notebook —
# we'll recompute them properly after splitting, using train data only
df = df.drop(columns=['user_avg_rating', 'attraction_avg_rating'])

print(df.shape)
df.head()

(52930, 22)


,TransactionId,UserId,VisitYear,VisitMonth,VisitModeId,AttractionId,Rating,ContinentId,RegionId,CountryId,...,AttractionCityId,AttractionTypeId,Attraction,AttractionAddress,AttractionType,Continent,Region,Country,CityName,user_visit_count
0,3,70456,2022,10,2,640,5,5,21,163,...,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Europe,Western Europe,United Kingdom,Guildford,1
1,8,7567,2022,10,4,640,5,2,8,48,...,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,America,Northern America,Canada,Ontario,1
2,9,79069,2022,10,3,640,5,2,9,54,...,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,America,South America,Brazil,Brazil,1
3,10,31019,2022,10,3,640,3,5,17,135,...,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Europe,Central Europe,Switzerland,Zurich,2
4,15,43611,2022,10,2,640,3,5,21,163,...,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Europe,Western Europe,United Kingdom,Manchester,3


In [19]:
feature_cols = ['VisitYear', 'VisitMonth', 'AttractionTypeId', 'Continent', 'Region',
                 'Country', 'VisitMode', 'user_visit_count']

# Ensure the dataset exists before using it
if 'df' not in globals():
    df = pd.read_csv('tourism_cleaned_engineered.csv')

X = df[feature_cols + ['UserId', 'AttractionId']].copy()  # keep IDs for later feature engineering
y = df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (42344, 10) Test: (10586, 10)


In [20]:
# Sirf TRAINING data se averages calculate karo
user_avg_rating_map = y_train.groupby(X_train['UserId']).mean()
attraction_avg_rating_map = y_train.groupby(X_train['AttractionId']).mean()
global_avg_rating = y_train.mean()  # naye/unseen users/attractions ke liye fallback

# Ab train aur test dono pe apply karo (map() use karke, dono training-only maps se)
X_train['user_avg_rating'] = X_train['UserId'].map(user_avg_rating_map).fillna(global_avg_rating)
X_train['attraction_avg_rating'] = X_train['AttractionId'].map(attraction_avg_rating_map).fillna(global_avg_rating)

X_test['user_avg_rating'] = X_test['UserId'].map(user_avg_rating_map).fillna(global_avg_rating)
X_test['attraction_avg_rating'] = X_test['AttractionId'].map(attraction_avg_rating_map).fillna(global_avg_rating)

# Ab UserId/AttractionId hata do — wo khud features nahi hain, sirf mapping ke liye chahiye the
X_train = X_train.drop(columns=['UserId', 'AttractionId'])
X_test = X_test.drop(columns=['UserId', 'AttractionId'])

print("Train nulls:\n", X_train.isnull().sum()[X_train.isnull().sum()>0])
print("\nTest nulls (expected — New users/attractions who were not on the train.):\n", X_test.isnull().sum()[X_test.isnull().sum()>0])

Train nulls:
 Series([], dtype: int64)

Test nulls (expected — New users/attractions who were not on the train.):
 Series([], dtype: int64)


In [21]:
X_train = X_train.drop(columns=['Continent', 'Region'])
X_test = X_test.drop(columns=['Continent', 'Region'])

cat_cols = ['Country', 'VisitMode']
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

print("Feature matrix shape:", X_train_enc.shape)

Feature matrix shape: (42344, 160)


In [22]:
#Model 1 — Linear Regression:

scaler = StandardScaler()
num_cols = ['VisitYear', 'VisitMonth', 'AttractionTypeId', 'user_visit_count', 'user_avg_rating', 'attraction_avg_rating']
X_train_scaled = X_train_enc.copy(); X_train_scaled[num_cols] = scaler.fit_transform(X_train_enc[num_cols])
X_test_scaled = X_test_enc.copy(); X_test_scaled[num_cols] = scaler.transform(X_test_enc[num_cols])

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)

print("Linear Regression:")
print("R2:", r2_score(y_test, pred_lr))
print("MAE:", mean_absolute_error(y_test, pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_lr)))

Linear Regression:
R2: -0.06829643167184063
MAE: 0.7343968326644826
RMSE: 1.0030647428340713


In [23]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train)
pred_rf = rf.predict(X_test_enc)

print("Random Forest:")
print("R2:", r2_score(y_test, pred_rf))
print("MAE:", mean_absolute_error(y_test, pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_rf)))

# XGBoost
xgb_reg = xgb.XGBRegressor(n_estimators=150, max_depth=6, random_state=42, n_jobs=-1)
xgb_reg.fit(X_train_enc, y_train)
pred_xgb = xgb_reg.predict(X_test_enc)

print("\nXGBoost:")
print("R2:", r2_score(y_test, pred_xgb))
print("MAE:", mean_absolute_error(y_test, pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_xgb)))

Random Forest:
R2: -0.06145489742406318
MAE: 0.7241962656160206
RMSE: 0.999847693689883

XGBoost:
R2: -0.06630527973175049
MAE: 0.7309074401855469
RMSE: 1.0021295522286893


In [24]:
feat_importance = pd.Series(xgb_reg.feature_importances_, index=X_train_enc.columns).sort_values(ascending=False)
print(feat_importance.head(15))

# Check karte hain user_avg_rating kitna reliable hai — kitne users ke paas kam visits hain
print("\nUser visit count distribution:")
print(df.groupby('UserId')['TransactionId'].count().describe())
print("\n% users with only 1 visit:", (df.groupby('UserId')['TransactionId'].count() == 1).mean() * 100)

user_avg_rating          0.601932
attraction_avg_rating    0.014686
Country_Thailand         0.009727
Country_Australia        0.009650
Country_Romania          0.008474
Country_Denmark          0.008278
Country_Turkey           0.008274
VisitMode_Family         0.007789
user_visit_count         0.007641
Country_Germany          0.007511
Country_United States    0.007505
Country_Switzerland      0.007293
Country_Cambodia         0.007243
Country_Canada           0.007215
Country_South Africa     0.007138
dtype: float32

User visit count distribution:
count    33530.000000
mean         1.578586
std          1.316680
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         59.000000
Name: TransactionId, dtype: float64

% users with only 1 visit: 68.33283626603041


In [25]:
X_train_enc = X_train_enc.drop(columns=['user_avg_rating'])
X_test_enc = X_test_enc.drop(columns=['user_avg_rating'])

print("New shape:", X_train_enc.shape)

# Dobara teenों models
lr = LinearRegression()
scaler = StandardScaler()
num_cols2 = ['VisitYear', 'VisitMonth', 'AttractionTypeId', 'user_visit_count', 'attraction_avg_rating']
X_train_s2 = X_train_enc.copy(); X_train_s2[num_cols2] = scaler.fit_transform(X_train_enc[num_cols2])
X_test_s2 = X_test_enc.copy(); X_test_s2[num_cols2] = scaler.transform(X_test_enc[num_cols2])
lr.fit(X_train_s2, y_train)
print("LR R2:", r2_score(y_test, lr.predict(X_test_s2)))

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train)
print("RF R2:", r2_score(y_test, rf.predict(X_test_enc)))

xgb_reg = xgb.XGBRegressor(n_estimators=150, max_depth=6, random_state=42, n_jobs=-1)
xgb_reg.fit(X_train_enc, y_train)
print("XGB R2:", r2_score(y_test, xgb_reg.predict(X_test_enc)))

New shape: (42344, 159)
LR R2: 0.09596286583339564
RF R2: 0.12787933434937238
XGB R2: 0.12525004148483276


In [26]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [150, 250],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1]
}
search = RandomizedSearchCV(xgb.XGBRegressor(random_state=42, n_jobs=-1), param_dist,
                             n_iter=4, scoring='r2', cv=3, random_state=42)
search.fit(X_train_enc, y_train)

print("Best params:", search.best_params_)
best_xgb = search.best_estimator_
print("Tuned R2:", r2_score(y_test, best_xgb.predict(X_test_enc)))

Best params: {'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.1}
Tuned R2: 0.12739449739456177


In [27]:
import joblib
joblib.dump(xgb_reg, 'best_regression_model.pkl')
joblib.dump(X_train_enc.columns.tolist(), 'regression_feature_columns.pkl')
joblib.dump(attraction_avg_rating_map, 'attraction_avg_rating_map.pkl')  # Predict page ke liye zaroori hoga
print("Saved.")

Saved.


## Regression Summary

**Final model:** XGBoost, R² = 0.125 (untuned baseline; hyperparameter tuning gave no meaningful improvement 0.127, within noise).

**Honest limitation:** This R² is low in absolute terms, and that's a genuine finding, not a bug. Two root causes, both traced back to EDA:
1. **`AttractionType` has a very narrow rating range (3.7–4.55 across all 17 types)**  the type of attraction barely differentiates rating, so there's limited signal in attraction-level features.
2. **68.3% of users have only a single visit in the dataset**, making any per-user pattern essentially unlearnable most of what determines an individual's rating (mood, specific experience that day) simply isn't captured in any available column.

**A real bug was caught and fixed during this process:** an initial version used `user_avg_rating` as a feature, which produced a **negative R²** (worse than predicting the mean) traced to that same 68.3% single-visit-user problem: for most users, "their average rating" was just one specific, unrelated data point, which the model overfit to on training data and failed to generalize from. Removing it turned a broken model into a modest-but-honest one.

**Takeaway for the recommendation system (next notebook):** given how sparse individual user history is, collaborative filtering will likely also be constrained by this same sparsity, and content-based filtering (which doesn't need per-user rating history) may prove more reliable here.